# Введение в MapReduce модель на Python


In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [ ]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [ ]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [ ]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [ ]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [ ]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [ ]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [ ]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [ ]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [ ]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [ ]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.991816935111749)),
 (1, np.float64(2.991816935111749)),
 (2, np.float64(2.991816935111749)),
 (3, np.float64(2.991816935111749)),
 (4, np.float64(2.991816935111749))]

## Inverted index

In [ ]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
print(output, '\n')


print(list(flatten(map(lambda x: MAP(*x), RECORDREADER()))) ,' map\n') #map


print(list(groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))),' map+shuffle\n') #map+shuffle ХОЧУ ПРО ЭТО СПРОСИТЬ ПРИ СДАЧЕ


print(list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))),' map+shuffle+reduce\n') #map+shuffle+reduce

[('what', ['0', '1']), ('it', ['0', '1', '2']), ('is', ['0', '1', '2']), ('banana', ['2']), ('a', ['2'])] 

[('what', '0'), ('it', '0'), ('is', '0'), ('it', '1'), ('is', '1'), ('what', '1'), ('banana', '2'), ('it', '2'), ('is', '2'), ('a', '2')]  map

[('what', ['0', '1']), ('it', ['0', '1', '2']), ('is', ['0', '1', '2']), ('banana', ['2']), ('a', ['2'])]  map+shuffle

[('what', ['0', '1']), ('it', ['0', '1', '2']), ('is', ['0', '1', '2']), ('banana', ['2']), ('a', ['2'])]  map+shuffle+reduce



## WordCount

In [ ]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [ ]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6)]),
 (1, [('a', 2), ('banana', 2), ('is', 18), ('it', 18), ('what', 10)])]

## TeraSort

In [ ]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.0076486150842939304)),
   (None, np.float64(0.01457336087116734)),
   (None, np.float64(0.02118335295155427)),
   (None, np.float64(0.04023803239537249)),
   (None, np.float64(0.042487008959853134)),
   (None, np.float64(0.07565879774516704)),
   (None, np.float64(0.07619503105776937)),
   (None, np.float64(0.13014411796016212)),
   (None, np.float64(0.17546760825726937)),
   (None, np.float64(0.18286861707531255)),
   (None, np.float64(0.1890086889334437)),
   (None, np.float64(0.19983298343503586)),
   (None, np.float64(0.24393660504168813)),
   (None, np.float64(0.3717761768972765)),
   (None, np.float64(0.37500231678389084)),
   (None, np.float64(0.45057670092323954)),
   (None, np.float64(0.457940038896152)),
   (None, np.float64(0.4855560948171719))]),
 (1,
  [(None, np.float64(0.53839529895589)),
   (None, np.float64(0.5526367214321335)),
   (None, np.float64(0.5598744251636685)),
   (None, np.float64(0.5642713959284413)),
   (None, np.float64(0.60876

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [ ]:
import numpy as np
from typing import Iterator

np.random.seed(23)
#a = np.random.randint(-1000, 1000, size=100).tolist()
a = [5, 7, 9, 2, 1]
maps = 3
reducers = 2

def INPUTFORMAT():
    split_size = int(np.ceil(len(a) / maps))
    for i in range(0, len(a), split_size):
        split = a[i:i+split_size]

        def RECORDREADER(split=split):
            for x in split:
                yield (None, x)

        yield RECORDREADER()

def MAP(_, x: int):
    yield ("max", x)

def REDUCE(key: str, values: Iterator[int]):
    yield (key, max(values))

COMBINER = REDUCE

out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=COMBINER)

result = []
for pid, part in out:
    result.extend(list(part))

print("max (MapReduceDistributed) =", dict(result)["max"])
print("max (python) =", max(a))

3 key-value pairs were sent over a network.
max (MapReduceDistributed) = 9
max (python) = 9


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [ ]:
np.random.seed(23)
a = np.random.randint(0, 100, size=100).tolist()

maps = 3
reducers = 2

def INPUTFORMAT():
    split_size = int(np.ceil(len(a) / maps))
    for i in range(0, len(a), split_size):
        split = a[i:i+split_size]

        def RECORDREADER(split=split):
            for x in split:
                yield (None, x)

        yield RECORDREADER()

def MAP(_, x: int):
    yield ("avg", (x, 1))

def REDUCE(key: str, values: Iterator[tuple]):
    sum = 0
    count = 0
    for s, c in values:
        sum += s
        count += c
    yield (key, sum / count)

COMBINER = lambda key, values: [
    (key, (sum(s for s, _ in values), sum(c for _, c in values)))
]

out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=COMBINER)

result = []
for pid, part in out:
    result.extend(list(part))

print("avg (MapReduceDistributed) =", dict(result)["avg"])
print("avg (python)               =", sum(a)/len(a))

3 key-value pairs were sent over a network.
avg (MapReduceDistributed) = 48.92
avg (python)               = 48.92


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [ ]:
from typing import Iterable, Iterator, List, Tuple, TypeVar, Callable, Dict
from itertools import groupby
from operator import itemgetter

K = TypeVar("K")
V = TypeVar("V")

def groupByKey_sort(pairs: Iterable[Tuple[K, V]],
                    keyfunc: Callable[[Tuple[K, V]], K] = lambda kv: kv[0]
                   ) -> Iterator[Tuple[K, List[V]]]:

    sorted_pairs = sorted(pairs, key=keyfunc)
    for k, group in groupby(sorted_pairs, key=keyfunc):
        yield (k, [v for _, v in group])


# Пример 1
pairs1 = [("a", 1), ("b", 2), ("a", 3), ("b", 4), ("c", 5), ("a", 10)]
print("Input:", pairs1)
print("groupByKey_sort:", list(groupByKey_sort(pairs1)))

# Пример 2
pairs2 = [(2, "x"), (1, "a"), (2, "y"), (1, "b"), (3, "z"), (2, "q")]
print("\nInput:", pairs2)
print("groupByKey_sort:", list(groupByKey_sort(pairs2)))

Input: [('a', 1), ('b', 2), ('a', 3), ('b', 4), ('c', 5), ('a', 10)]
groupByKey_sort: [('a', [1, 3, 10]), ('b', [2, 4]), ('c', [5])]

Input: [(2, 'x'), (1, 'a'), (2, 'y'), (1, 'b'), (3, 'z'), (2, 'q')]
groupByKey_sort: [(1, ['a', 'b']), (2, ['x', 'y', 'q']), (3, ['z'])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [ ]:
import numpy as np

a = np.random.randint(0, 10, size=100).tolist()

maps = 3
reducers = 2

def INPUTFORMAT():
    split_size = int(np.ceil(len(a) / maps))
    for i in range(0, len(a), split_size):
        split = a[i:i+split_size]

        def RECORDREADER(split=split):
            for x in split:
                yield (None, x)

        yield RECORDREADER()

def MAP(_, x):
    yield (x, 1)

def COMBINER(x, values):
    yield (x, 1)

def REDUCE(x, values):
    yield (x, x)

out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=COMBINER)

ans = []
for pid, part in out:
    for k, v in part:
        ans.append(v)

ans = sorted(ans)

print("distinct =", ans)
print("check =", sorted(set(a)))

29 key-value pairs were sent over a network.
distinct = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
check = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [ ]:
import numpy as np

R = [
    ("Ivan", 20, "Astana"),
    ("Ksusha", 20, "Moscow"),
    ("Dima", 15, "New-York"),
    ("Eva", 30, "Rumochka"),
]

def C(t):
    name, age, city = t
    return age >= 18

maps = 3
reducers = 2

def INPUTFORMAT():
    split_size = int(np.ceil(len(R) / maps))
    for i in range(0, len(R), split_size):
        split = R[i:i+split_size]

        def RECORDREADER(split=split):
            for t in split:
                yield (None, t)

        yield RECORDREADER()

def MAP(_, t):
    if C(t):
        yield (t, t)

def REDUCE(key, values):
    for v in values:
        yield (key, v)

out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

ans = []
for pid, part in out:
    for k, v in part:
        ans.append(v)

print("RESULT =", ans)

3 key-value pairs were sent over a network.
RESULT = [('Eva', 30, 'Rumochka'), ('Ivan', 20, 'Astana'), ('Ksusha', 20, 'Moscow')]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [ ]:
import numpy as np

R = [
    ("Ivan", 20, "Astana"),
    ("Ksusha", 20, "Moscow"),
    ("Dima", 15, "New-York"),
    ("Eva", 30, "Rumochka"),
]

S = [1, 2]

maps = 3
reducers = 2

def INPUTFORMAT():
    split_size = int(np.ceil(len(R) / maps))
    for i in range(0, len(R), split_size):
        split = R[i:i+split_size]

        def RECORDREADER(split=split):
            for t in split:
                yield (None, t)

        yield RECORDREADER()

def MAP(_, t):
    t2 = tuple(t[i] for i in S)
    yield (t2, t2)

def REDUCE(t2, values):
    yield (t2, t2)

out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

ans = []
for pid, part in out:
    for k, v in part:
        ans.append(v)

print("R =", R)
print("S =", S)
print("Projection =", sorted(ans))

4 key-value pairs were sent over a network.
R = [('Ivan', 20, 'Astana'), ('Ksusha', 20, 'Moscow'), ('Dima', 15, 'New-York'), ('Eva', 30, 'Rumochka')]
S = [1, 2]
Projection = [(15, 'New-York'), (20, 'Astana'), (20, 'Moscow'), (30, 'Rumochka')]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [ ]:
import numpy as np

R1 = [
    ("Ivan", 20, "Astana"),
    ("Ksusha", 20, "Moscow"),
    ("Dima", 15, "New-York"),
    ("Eva", 30, "Rumochka"),
]

R2 = [
    ("Ivan", 20, "Astana"),
    ("Alina", 25, "Almaty"),
    ("Dima", 15, "New-York"),
    ("Petr", 40, "Moscow"),
]

maps = 3
reducers = 2

ALL = R1 + R2

def INPUTFORMAT():
    split_size = int(np.ceil(len(ALL) / maps))
    for i in range(0, len(ALL), split_size):
        split = ALL[i:i+split_size]

        def RECORDREADER(split=split):
            for t in split:
                yield (None, t)

        yield RECORDREADER()

def MAP(_, t):
    yield (t, t)

def REDUCE(t, values):
    yield (t, t)

out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

ans = []
for pid, part in out:
    for k, v in part:
        ans.append(v)

print("R1 =", R1)
print("R2 =", R2)
print("UNION =", sorted(ans))

8 key-value pairs were sent over a network.
R1 = [('Ivan', 20, 'Astana'), ('Ksusha', 20, 'Moscow'), ('Dima', 15, 'New-York'), ('Eva', 30, 'Rumochka')]
R2 = [('Ivan', 20, 'Astana'), ('Alina', 25, 'Almaty'), ('Dima', 15, 'New-York'), ('Petr', 40, 'Moscow')]
UNION = [('Alina', 25, 'Almaty'), ('Dima', 15, 'New-York'), ('Eva', 30, 'Rumochka'), ('Ivan', 20, 'Astana'), ('Ksusha', 20, 'Moscow'), ('Petr', 40, 'Moscow')]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [ ]:
import numpy as np

R1 = [
    ("Ivan", 20, "Astana"),
    ("Ksusha", 20, "Moscow"),
    ("Dima", 15, "New-York"),
    ("Eva", 30, "Rumochka"),
]

R2 = [
    ("Ivan", 20, "Astana"),
    ("Alina", 25, "Almaty"),
    ("Dima", 15, "New-York"),
    ("Petr", 40, "Moscow"),
]

maps = 3
reducers = 2

ALL = [("R1", t) for t in R1] + [("R2", t) for t in R2]

def INPUTFORMAT():
    split_size = int(np.ceil(len(ALL) / maps))
    for i in range(0, len(ALL), split_size):
        split = ALL[i:i+split_size]

        def RECORDREADER(split=split):
            for tag, t in split:
                yield (tag, t)

        yield RECORDREADER()

def MAP(tag, t):
    yield (t, tag)

def REDUCE(t, tags):
    s = set(tags)
    if ("R1" in s) and ("R2" in s):
        yield (t, t)

out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

ans = []
for pid, part in out:
    for k, v in part:
        ans.append(v)

print("R1 =", R1)
print("R2 =", R2)
print("INTERSECTION =", sorted(ans))

8 key-value pairs were sent over a network.
R1 = [('Ivan', 20, 'Astana'), ('Ksusha', 20, 'Moscow'), ('Dima', 15, 'New-York'), ('Eva', 30, 'Rumochka')]
R2 = [('Ivan', 20, 'Astana'), ('Alina', 25, 'Almaty'), ('Dima', 15, 'New-York'), ('Petr', 40, 'Moscow')]
INTERSECTION = [('Dima', 15, 'New-York'), ('Ivan', 20, 'Astana')]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [ ]:
import numpy as np

R = [
    ("Ivan", 20, "Astana"),
    ("Ksusha", 20, "Moscow"),
    ("Dima", 15, "New-York"),
    ("Eva", 30, "Rumochka"),
]

S = [
    ("Ivan", 20, "Astana"),
    ("Dima", 15, "New-York"),
    ("Petr", 40, "Moscow"),
]

maps = 3
reducers = 2

ALL = [("R", t) for t in R] + [("S", t) for t in S]

def INPUTFORMAT():
    split_size = int(np.ceil(len(ALL) / maps))
    for i in range(0, len(ALL), split_size):
        split = ALL[i:i+split_size]

        def RECORDREADER(split=split):
            for tag, t in split:
                yield (tag, t)

        yield RECORDREADER()

def MAP(tag, t):
    yield (t, tag)

def REDUCE(t, tags):
    tags = set(tags)
    if ("R" in tags) and ("S" not in tags):
        yield (t, t)

out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

ans = []
for pid, part in out:
    for k, v in part:
        ans.append(v)

print("R =", R)
print("S =", S)
print("R \\ S =", sorted(ans))

7 key-value pairs were sent over a network.
R = [('Ivan', 20, 'Astana'), ('Ksusha', 20, 'Moscow'), ('Dima', 15, 'New-York'), ('Eva', 30, 'Rumochka')]
S = [('Ivan', 20, 'Astana'), ('Dima', 15, 'New-York'), ('Petr', 40, 'Moscow')]
R \ S = [('Eva', 30, 'Rumochka'), ('Ksusha', 20, 'Moscow')]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [ ]:
import numpy as np

R = [
    ("Ivan", 20),
    ("Ksusha", 20),
    ("Dima", 15),
    ("Eva", 30),
]

S = [
    (20, "Moscow"),
    (15, "New-York"),
    (20, "Astana"),
    (99, "Nowhere"),
]

maps = 3
reducers = 2

ALL = [("R", t) for t in R] + [("S", t) for t in S]

def INPUTFORMAT():
    split_size = int(np.ceil(len(ALL) / maps))
    for i in range(0, len(ALL), split_size):
        split = ALL[i:i+split_size]

        def RECORDREADER(split=split):
            for tag, t in split:
                yield (tag, t)

        yield RECORDREADER()

def MAP(tag, t):
    if tag == "R":
        a, b = t
        yield (b, ("R", a))
    else:
        b, c = t
        yield (b, ("S", c))

def REDUCE(b, values):
    A = []
    C = []

    for kind, x in values:
        if kind == "R":
            A.append(x)
        else:
            C.append(x)

    for a in A:
        for c in C:
            yield (None, (a, b, c))

out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

ans = []
for pid, part in out:
    for k, v in part:
        ans.append(v)

print("R =", R)
print("S =", S)
print("JOIN =", sorted(ans))

8 key-value pairs were sent over a network.
R = [('Ivan', 20), ('Ksusha', 20), ('Dima', 15), ('Eva', 30)]
S = [(20, 'Moscow'), (15, 'New-York'), (20, 'Astana'), (99, 'Nowhere')]
JOIN = [('Dima', 15, 'New-York'), ('Ivan', 20, 'Astana'), ('Ivan', 20, 'Moscow'), ('Ksusha', 20, 'Astana'), ('Ksusha', 20, 'Moscow')]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [ ]:
import numpy as np

T = [
    ("Ivan", 5, "x"),
    ("Ivan", 7, "y"),
    ("Ivan", 2, "z"),
    ("Dima", 10, "q"),
    ("Dima", 1, "w"),
    ("Eva", 3, "p"),
]

maps = 3
reducers = 2

def INPUTFORMAT():
    split_size = int(np.ceil(len(T) / maps))
    for i in range(0, len(T), split_size):
        split = T[i:i+split_size]

        def RECORDREADER(split=split):
            for row in split:
                yield (None, row)

        yield RECORDREADER()

def MAP(_, row):
    a, b, c = row
    yield (a, b)

def REDUCE(a, bs):
    s = 0
    for b in bs:
        s += b
    yield (a, s)

COMBINER = REDUCE

out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=COMBINER)

ans = []
for pid, part in out:
    for k, v in part:
        ans.append((k, v))

print("T =", T)
print("GROUP SUM =", sorted(ans))

5 key-value pairs were sent over a network.
T = [('Ivan', 5, 'x'), ('Ivan', 7, 'y'), ('Ivan', 2, 'z'), ('Dima', 10, 'q'), ('Dima', 1, 'w'), ('Eva', 3, 'p')]
GROUP SUM = [('Dima', 11), ('Eva', 3), ('Ivan', 14)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [ ]:
import numpy as np


m, n = 4, 5
np.random.seed(1)

A = np.random.randint(0, 5, size=(m, n))
x = np.random.randint(0, 5, size=n)

A_records = []
for i in range(m):
    for j in range(n):
        A_records.append(("A", i, j, int(A[i, j])))

X_records = []
for j in range(n):
    X_records.append(("X", j, int(x[j])))

ALL = A_records + X_records

maps = 3
reducers = 2

def INPUTFORMAT_FROM_LIST(data_list):
    def INPUTFORMAT():
        split_size = int(np.ceil(len(data_list) / maps))
        for k in range(0, len(data_list), split_size):
            split = data_list[k:k+split_size]

            def RECORDREADER(split=split):
                for rec in split:
                    yield (None, rec)

            yield RECORDREADER()
    return INPUTFORMAT

def MAP1(_, rec):
    if rec[0] == "A":
        _, i, j, aij = rec
        yield (j, ("A", i, aij))
    else:
        _, j, xj = rec
        yield (j, ("X", xj))

def REDUCE1(j, values):
    xj = None
    a_list = []

    for item in values:
        if item[0] == "X":
            xj = item[1]
        else:
            _, i, aij = item
            a_list.append((i, aij))

    if xj is None:
        return

    for i, aij in a_list:
        yield (i, aij * xj)

out1 = MapReduceDistributed(INPUTFORMAT_FROM_LIST(ALL), MAP1, REDUCE1)

partials = []
for pid, part in out1:
    for i, contrib in part:
        partials.append((i, contrib))

print("partials sample:", partials[:10])

def INPUTFORMAT2():
    split_size = int(np.ceil(len(partials) / maps))
    for k in range(0, len(partials), split_size):
        split = partials[k:k+split_size]

        def RECORDREADER(split=split):
            for i, contrib in split:
                yield (i, contrib)

        yield RECORDREADER()

def MAP2(i, contrib):
    yield (i, contrib)

def REDUCE2(i, contribs):
    s = 0
    for v in contribs:
        s += v
    yield (i, s)

COMBINER2 = REDUCE2

out2 = MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2, COMBINER=COMBINER2)

y_list = [0] * m
for pid, part in out2:
    for i, val in part:
        y_list[i] = val

print("A=\n", A)
print("x=", x)
print("y (MapReduce) =", y_list)
print("y (numpy)     =", (A @ x).tolist())

25 key-value pairs were sent over a network.
partials sample: [(0, 12), (1, 0), (2, 4), (3, 12), (0, 0), (1, 1), (2, 4), (3, 2), (0, 3), (1, 4)]
12 key-value pairs were sent over a network.
A=
 [[3 4 0 1 3]
 [0 0 1 4 4]
 [1 2 4 2 4]
 [3 4 2 4 2]]
x= [4 1 1 0 1]
y (MapReduce) = [19, 5, 14, 20]
y (numpy)     = [19, 5, 14, 20]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [ ]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [ ]:
import numpy as np

I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I, J)
big_mat   = np.random.rand(J, K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j, k), big_mat[j, k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
    yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  (i, k) = key
  s = 0.0
  for v in values:
    s += v
  yield ((i, k), s)

out = list(MapReduce(RECORDREADER, MAP, REDUCE))

P = np.zeros((I, K))
for (i, k), val in out:
  P[i, k] = val

Проверьте своё решение

In [ ]:
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output) + 1
  K = max(k for ((i,k), vw) in reduce_output) + 1
  mat = np.empty(shape=(I, K))
  for ((i,k), vw) in reduce_output:
    mat[i, k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution))

True

In [ ]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [ ]:
import numpy as np

def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x),
                     groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

np.random.seed(0)
I, J, K = 2, 3, 4*10
M = np.random.rand(I, J)
N = np.random.rand(J, K)

def RECORDREADER():
  for i in range(I):
    for j in range(J):
      yield (("M", i, j), M[i, j])
  for j in range(J):
    for k in range(K):
      yield (("N", j, k), N[j, k])

def MAP(k1, v1):
  tag = k1[0]
  if tag == "M":
    _, i, j = k1
    v = v1
    for k in range(K):
      yield ((i, k), ("M", j, v))
  else:
    _, j, k = k1
    w = v1
    for i in range(I):
      yield ((i, k), ("N", j, w))

def REDUCE(key, values):
  i, k = key
  mvals = {}
  nvals = {}

  for typ, j, val in values:
    if typ == "M":
      mvals[j] = mvals.get(j, 0.0) + val
    else:
      nvals[j] = nvals.get(j, 0.0) + val

  s = 0.0
  for j, v in mvals.items():
    w = nvals.get(j, 0.0)
    s += v * w

  yield ((i, k), s)

reference_solution = np.matmul(M, N)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I2 = max(i for ((i,k), vw) in reduce_output)+1
  K2 = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I2, K2))
  for ((i,k), vw) in reduce_output:
    mat[i, k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution))

True

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [ ]:
import numpy as np

np.random.seed(0)
I, J, K = 2, 3, 4*10
M = np.random.rand(I, J)
N = np.random.rand(J, K)

maps = 2
reducers = 2

def INPUTFORMAT():
  def RR_M():
    for i in range(I):
      for j in range(J):
        yield (("M", i, j), M[i, j])

  def RR_N():
    for j in range(J):
      for k in range(K):
        yield (("N", j, k), N[j, k])

  yield RR_M()
  yield RR_N()

def MAP(k1, v1):
  tag = k1[0]
  if tag == "M":
    _, i, j = k1
    v = v1
    for k in range(K):
      yield ((i, k), ("M", j, v))
  else:
    _, j, k = k1
    w = v1
    for i in range(I):
      yield ((i, k), ("N", j, w))

def REDUCE(key, values):
  i, k = key
  mvals = {}
  nvals = {}

  for typ, j, val in values:
    if typ == "M":
      mvals[j] = mvals.get(j, 0.0) + val
    else:
      nvals[j] = nvals.get(j, 0.0) + val

  s = 0.0
  for j, v in mvals.items():
    s += v * nvals.get(j, 0.0)

  yield ((i, k), s)

reference_solution = np.matmul(M, N)
out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

reduce_output = []
for pid, part in out:
  reduce_output.extend(list(part))

def asmatrix2(reduce_output):
  I2 = max(i for ((i,k), vw) in reduce_output)+1
  K2 = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty((I2, K2))
  for ((i,k), vw) in reduce_output:
    mat[i, k] = vw
  return mat

np.allclose(reference_solution, asmatrix2(reduce_output))

480 key-value pairs were sent over a network.


True

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [ ]:
import numpy as np

np.random.seed(0)
I, J, K = 2, 3, 40
M = np.random.rand(I, J)
N = np.random.rand(J, K)

maps = 3
reducers = 2

M_readers = 2
N_readers = 3

mode = "partitioned"

def INPUTFORMAT():
  for rid in range(M_readers):
    def RR_M(rid=rid):
      for i in range(I):
        for j in range(J):
          if mode == "partitioned":
            if (i*J + j) % M_readers != rid:
              continue
            yield (("M", i, j), M[i, j])
          else:
            if np.random.rand() < 0.6:
              yield (("M", i, j), M[i, j])
    yield RR_M()

  for rid in range(N_readers):
    def RR_N(rid=rid):
      for j in range(J):
        for k in range(K):
          if mode == "partitioned":
            if (j*K + k) % N_readers != rid:
              continue
            yield (("N", j, k), N[j, k])
          else:
            if np.random.rand() < 0.6:
              yield (("N", j, k), N[j, k])
    yield RR_N()

def MAP(k1, v1):
  tag = k1[0]
  if tag == "M":
    _, i, j = k1
    v = v1
    for k in range(K):
      yield ((i, k), ("M", j, v))
  else:
    _, j, k = k1
    w = v1
    for i in range(I):
      yield ((i, k), ("N", j, w))

def REDUCE(key, values):
  i, k = key
  mvals = {}
  nvals = {}

  for typ, j, val in values:
    if typ == "M":
      mvals[j] = mvals.get(j, 0.0) + val
    else:
      nvals[j] = nvals.get(j, 0.0) + val

  s = 0.0
  for j, v in mvals.items():
    s += v * nvals.get(j, 0.0)

  yield ((i, k), s)

out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

reduce_output = []
for pid, part in out:
  reduce_output.extend(list(part))

def asmatrix2(reduce_output):
  I2 = max(i for ((i,k), vw) in reduce_output)+1
  K2 = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty((I2, K2))
  for ((i,k), vw) in reduce_output:
    mat[i, k] = vw
  return mat

P_mr = asmatrix2(reduce_output)
P_ref = M @ N

print("mode =", mode)
print("allclose(MR, numpy) =", np.allclose(P_mr, P_ref))

480 key-value pairs were sent over a network.
mode = partitioned
allclose(MR, numpy) = True
